<p style="align: center;"><img src="https://static.tildacdn.com/tild6636-3531-4239-b465-376364646465/Deep_Learning_School.png" width="400"></p>

# Глубокое обучение. Часть 2
# Домашнее задание по теме "Механизм внимания"

Это домашнее задание проходит в формате peer-review. Это означает, что его будут проверять ваши однокурсники. Поэтому пишите разборчивый код, добавляйте комментарии и пишите выводы после проделанной работы.

В этом задании вы будете решать задачу классификации математических задач по темам (многоклассовая классификация) с помощью Transformer.

В качестве датасета возьмем датасет математических задач по разным темам. Нам необходим следующий файл:

[Файл с классами](https://docs.google.com/spreadsheets/d/13YIbphbWc62sfa-bCh8MLQWKizaXbQK9/edit?usp=drive_link&ouid=104379615679964018037&rtpof=true&sd=true)

**Hint:** не перезаписывайте модели, которые вы получите на каждом из этапов этого дз. Они ещё понадобятся.

### Задание 1 (2 балла)

Напишите кастомный класс для модели трансформера для задачи классификации, использующей в качествке backbone какую-то из моделей huggingface.

Т.е. конструктор класса должен принимать на вход название модели и подгружать её из huggingface, а затем использовать в качестве backbone (достаточно возможности использовать в качестве backbone те модели, которые упомянуты в последующих пунктах)

In [14]:
from IPython.display import clear_output

In [15]:
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(palette='summer')
clear_output()

In [16]:
!pip install accelerate -U
clear_output()

In [17]:
!pip install datasets
clear_output()

In [18]:
!pip install transformers
clear_output()

In [19]:
!pip install evaluate
clear_output()

In [20]:
import transformers
from datasets import load_dataset
import evaluate

In [21]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [22]:
from transformers import AutoModel

In [23]:
class TransformerClassificationModel(nn.Module):
    def __init__(self, base_transformer_model: str, num_classes: int):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(base_transformer_model)
        hidden_size = self.backbone.config.hidden_size
        self.classifier = nn.Linear(in_features=hidden_size, out_features=num_classes)

    def forward(self, input_ids, attention_mask):
        backbone_outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = backbone_outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_embedding)
        outputs = {"logits": logits}
        return outputs

### Задание 2 (1 балл)

Напишите функцию заморозки backbone у модели (если необходимо, возвращайте из функции модель)

In [24]:
def freeze_backbone_function(model: TransformerClassificationModel):
    for param in model.backbone.parameters():
        param.requires_grad = False

    return model

### Задание 3 (2 балла)

Напишите функцию, которая будет использована для тренировки (дообучения) трансформера (TransformerClassificationModel). Функция должна поддерживать обучение с замороженным и размороженным backbone.

In [25]:
import copy

def train_transformer(transformer_model, train_dataloader, epochs=5, freeze_backbone=True, device=device):
    model = copy.deepcopy(transformer_model)
    if freeze_backbone:
        for param in model.backbone.parameters():
            param.requires_grad = False
    else:
        for param in model.backbone.parameters():
            param.requires_grad = True

    model.to(device)

    criterion = nn.CrossEntropyLoss()

    trainable_params = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = torch.optim.AdamW(trainable_params, lr=2e-5)

    model.train()

    for epoch in range(epochs):
        total_loss = 0
        for batch in tqdm(train_dataloader, desc=f"epoch {epoch+1}/{epochs}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs['logits']

            loss = criterion(logits, labels)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        print(f"epoch {epoch + 1}: average loss: {total_loss / len(train_dataloader)} damn that's a lot actually")

    done_model = model
    return done_model

In [26]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

df = pd.read_csv("data_problems_translated.csv")

df = df.dropna(subset=['problem_text', 'topic'])

unique_topics = df['topic'].unique()
topic2id = {topic: i for i, topic in enumerate(unique_topics)}

df['label_id'] = df['topic'].map(topic2id)

num_classes = len(unique_topics)
print(f"Количество классов (тем): {num_classes}")

hf_dataset = Dataset.from_pandas(df)

model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["problem_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.rename_column("label_id", "labels")

tokenized_dataset.set_format("torch", columns=['input_ids', 'attention_mask', 'labels'])

train_dataloader = DataLoader(
    tokenized_dataset,
    shuffle=True,
    batch_size=16
)

print("DataLoader успешно создан!")

Количество классов (тем): 7


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/5273 [00:00<?, ? examples/s]

DataLoader успешно создан!


### Задание 4 (1 балл)

Проверьте вашу функцию из предыдущего пункта, дообучив двумя способами
*cointegrated/rubert-tiny2* из huggingface.

In [28]:
num_classes = 7

rubert_tiny_transformer_model = TransformerClassificationModel(
    base_transformer_model="cointegrated/rubert-tiny2",
    num_classes=num_classes
)

rubert_tiny_finetuned_with_freezed_backbone = train_transformer(
    transformer_model=rubert_tiny_transformer_model,
    train_dataloader=train_dataloader,
    freeze_backbone=True
)

rubert_tiny_transformer_model = TransformerClassificationModel(
    base_transformer_model="cointegrated/rubert-tiny2",
    num_classes=num_classes
)

rubert_tiny_full_finetuned = train_transformer(
    transformer_model=rubert_tiny_transformer_model,
    train_dataloader=train_dataloader,
    freeze_backbone=False
)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1/5:   0%|          | 0/330 [00:00<?, ?it/s]

epoch 1: average loss: 1.866347445863666 damn that's a lot actually


epoch 2/5:   0%|          | 0/330 [00:00<?, ?it/s]

...

epoch 3: average loss: 0.9254939042257541 damn that's a lot actually


epoch 4/5:   0%|          | 0/330 [00:00<?, ?it/s]

epoch 4: average loss: 0.8514023397004966 damn that's a lot actually


epoch 5/5:   0%|          | 0/330 [00:00<?, ?it/s]

epoch 5: average loss: 0.7956498550646233 damn that's a lot actually


In [29]:
import torch

torch.save(
    rubert_tiny_finetuned_with_freezed_backbone.state_dict(),
    "rubert_tiny_freezed.pth"
)

torch.save(
    rubert_tiny_full_finetuned.state_dict(),
    "rubert_tiny_full.pth"
)

print("веса моделей успешно сохранены!")

веса моделей успешно сохранены!
